#### Gaming A/B Test Data

**Project goal**  
Build end-to-end A/B testing analytics pipeline for a casual/instant game  

**Current experiment**  
- Variant A = control (standard daily login rewards)  
- Variant B = treatment (new streak multipliers + better rewards)

**Expected outcome**  
B should show realistic positive lifts in retention, engagement & monetization

#### 1. Imports & seed

In [14]:
import pandas as pd
import numpy as np

np.random.seed(42)

print("Libraries loaded")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

Libraries loaded
NumPy version: 1.26.3
Pandas version: 2.3.3


#### 2. Parameters

In [4]:
N_USERS = 100_000
AB_SPLIT = 0.50           

print(f"Target users     : {N_USERS:,}")
print(f"A/B assignment   : {AB_SPLIT*100:.0f}% A  /  {(1-AB_SPLIT)*100:.0f}% B")

Target users     : 100,000
A/B assignment   : 50% A  /  50% B


#### 3. Core columns

In [5]:
# Unique user identifiers
user_ids = np.arange(1, N_USERS + 1)

# A/B assignment
variants = np.random.choice(['A', 'B'], size=N_USERS, p=[1 - AB_SPLIT, AB_SPLIT])

# Realistic install dates: spread across ~2.5 years (2023–2025)
start_date = pd.to_datetime('2023-04-01')
end_date   = pd.to_datetime('2025-10-31')

# Uniform random timestamps between start and end
time_delta_seconds = (end_date - start_date).total_seconds()
random_seconds = np.random.uniform(0, time_delta_seconds, N_USERS)

install_dates = start_date + pd.to_timedelta(random_seconds, unit='s')

# Sort chronologically (most common in raw event data)
install_dates = pd.Series(install_dates).sort_values().reset_index(drop=True)

# Quick sanity check
print("Install date range:", install_dates.min().date(), "–", install_dates.max().date())
print("Unique install days:", install_dates.dt.date.nunique())
print("\nA/B balance:\n", pd.Series(variants).value_counts())

Install date range: 2023-04-01 – 2025-10-30
Unique install days: 944

A/B balance:
 B    50066
A    49934
Name: count, dtype: int64


#### 4. Business metrics

In [8]:
# Retention (binary 0/1)
d1_retention  = np.random.binomial(1, 0.45 + (variants == 'B') * 0.055, N_USERS)
d7_retention  = np.random.binomial(1, 0.25 + (variants == 'B') * 0.075, N_USERS)
d30_retention = np.random.binomial(1, 0.10 + (variants == 'B') * 0.055, N_USERS)

# Revenue (exponential distribution + uplift)
base_rev = np.random.exponential(scale=0.50, size=N_USERS)
rev_multiplier = 1 + (variants == 'B') * 0.15 + np.random.normal(0, 0.04, N_USERS)
revenue = np.maximum(base_rev * rev_multiplier, 0).round(2)

# Sessions on day 1 
sessions_d1 = np.maximum(
    np.random.poisson(2.6 * (1 + (variants == 'B') * 0.18), N_USERS),
    1
)

# Levels completed in first 7 days
levels_d7 = np.random.poisson(15 * (1 + (variants == 'B') * 0.24), N_USERS)

# Ever made a purchase
is_payer = np.random.binomial(1, 0.029 + (variants == 'B') * 0.021, N_USERS)

print("Metrics generated (first 5 users):")
print("D7 retention sample:", d7_retention[:5])
print("Revenue sample:", revenue[:5])
print("Sessions D1 sample:", sessions_d1[:5])

Metrics generated (first 5 users):
D7 retention sample: [1 0 0 0 0]
Revenue sample: [0.6  0.93 0.45 0.5  0.03]
Sessions D1 sample: [1 5 1 5 1]


#### 5. Create final DataFrame

In [9]:
df = pd.DataFrame({
    'user_id':      user_ids,
    'variant':      variants,
    'install_date': install_dates,
    'd1_retention':  d1_retention,
    'd7_retention':  d7_retention,
    'd30_retention': d30_retention,
    'revenue':       revenue,
    'sessions_d1':   sessions_d1,
    'levels_d7':     levels_d7,
    'is_payer':      is_payer
})

# Save raw data
save_path = "../data/raw_data/gaming_ab_test_raw_100k_v2.csv"
df.to_csv(save_path, index=False)

print(f"Dataset saved → {save_path}")
print("Shape:", df.shape)
print("\nFirst 4 rows:")
display(df.head(4))

Dataset saved → ../data/raw_data/gaming_ab_test_raw_100k_v2.csv
Shape: (100000, 10)

First 4 rows:


,user_id,variant,install_date,d1_retention,d7_retention,d30_retention,revenue,sessions_d1,levels_d7,is_payer
0,1,A,2023-04-01 00:06:29.751009121,1,1,0,0.60,1,14,0
1,2,B,2023-04-01 00:08:56.022639616,1,0,1,0.93,5,22,0
2,3,B,2023-04-01 00:21:26.287279752,1,0,0,0.45,1,18,0
3,4,B,2023-04-01 00:35:40.336226844,0,0,0,0.50,5,25,0


In [10]:
df.tail(3)

,user_id,variant,install_date,d1_retention,d7_retention,d30_retention,revenue,sessions_d1,levels_d7,is_payer
99997,99998,B,2025-10-30 23:29:47.142028928,1,0,0,0.56,4,17,0
99998,99999,A,2025-10-30 23:36:52.207559824,0,0,0,0.68,1,6,0
99999,100000,A,2025-10-30 23:45:34.209441796,0,0,0,0.18,3,13,0


#### 6. Final business summary

In [13]:
summary = df.groupby('variant')[['d1_retention', 'd7_retention', 'd30_retention',
                                 'revenue', 'sessions_d1', 'levels_d7', 'is_payer']].mean().round(4)

print("Average performance by variant:")
display(summary)

uplift_pct = ((summary.loc['B'] / summary.loc['A'] - 1) * 100).round(1)
print("\nUplift when using Variant B (%):")
print(uplift_pct)

Average performance by variant:


,d1_retention,d7_retention,d30_retention,revenue,sessions_d1,levels_d7,is_payer
variant,,,,,,,
A,0.4462,0.2509,0.1015,0.4962,2.6766,14.9861,0.0276
B,0.5004,0.3226,0.1538,0.5769,3.1096,18.5790,0.0503



Uplift when using Variant B (%):
d1_retention     12.1
d7_retention     28.6
d30_retention    51.5
revenue          16.3
sessions_d1      16.2
levels_d7        24.0
is_payer         82.2
dtype: float64
